## ML Flow Experiment Tracking

### Importing Libraries

In [51]:
import pandas as pd
import joblib
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import mlflow
from mlflow.models import infer_signature

### Importing DataSets

In [52]:
df_train = pd.read_csv("../data/ml_ready/train.csv")
df_valid = pd.read_csv("../data/ml_ready/valid.csv")
df_test = pd.read_csv("../data/ml_ready/test.csv")

In [53]:
print(df_train.shape)
print(df_valid.shape)
print(df_test.shape)

(43534, 39)
(21762, 39)
(19948, 39)


In [54]:
X_train = df_train.drop("next_monthly_revenue", axis=1)
X_valid = df_valid.drop("next_monthly_revenue", axis=1)
X_test = df_test.drop("next_monthly_revenue", axis=1)

y_train = df_train["next_monthly_revenue"]
y_valid = df_valid["next_monthly_revenue"]
y_test = df_test["next_monthly_revenue"]

In [55]:
loaded_scaler = joblib.load("../models/scaler.joblib")

X_train_scaled = loaded_scaler.transform(X_train)
X_valid_scaled = loaded_scaler.transform(X_valid)
X_test_scaled = loaded_scaler.transform(X_test)

### Experiment Tracking - MLFLOW

In [56]:
rf_loaded_model = joblib.load("../models/random_forest_model.joblib")
light_loaded_model = joblib.load("../models/light_gbm_model.joblib")

#### Predicting on Test Data

In [57]:
rf_y_pred = rf_loaded_model.predict(X_test_scaled)
lgbm_y_pred = light_loaded_model.predict(X_test_scaled)

#### Calculating Metrics for Test Data

In [58]:
rf_mae = mean_absolute_error(y_test, rf_y_pred)
rf_rmse = mean_squared_error(y_test, rf_y_pred)
rf_r2 = r2_score(y_test, rf_y_pred)

rf_metrics_dict = {
    "MAE": rf_mae,
    "RMSE": rf_rmse,
    "R2 Score": rf_r2
}

lgbm_mae = mean_absolute_error(y_test, lgbm_y_pred)
lgbm_rmse = mean_squared_error(y_test, lgbm_y_pred)
lgbm_r2 = r2_score(y_test, lgbm_y_pred)

lgbm_metrics_dict = {
    "MAE": lgbm_mae,
    "RMSE": lgbm_rmse,
    "R2 Score": lgbm_r2
}

#### Experiment Tracking

In [59]:
mlflow.set_experiment("Predictive Sales Intelligence Platform")
mlflow.set_tracking_uri("http://127.0.0.1:5000/")

In [60]:
with mlflow.start_run():
    ## Logging Parameters, models, Metrics
    mlflow.set_tag("Model", "Random Forest")
    # Fix: Remove the string title. Pass only the dictionary
    mlflow.log_params(rf_loaded_model.get_params())
    mlflow.log_metrics(rf_metrics_dict)
    mlflow.log_artifact("../models/scaler.joblib")
    mlflow.log_artifact("../models/random_forest_model.joblib")

🏃 View run colorful-shrimp-216 at: http://127.0.0.1:5000/#/experiments/1/runs/a3919db875794e66a769c0fadf4cb57e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [61]:
with mlflow.start_run():
    ## Logging Parameters, models, Metrics
    mlflow.set_tag("Model", "Light GBM")
    # Fix: Remove the string title. Pass only the dictionary
    mlflow.log_params(light_loaded_model.get_params())
    mlflow.log_metrics(lgbm_metrics_dict)
    mlflow.log_artifact("../models/scaler.joblib")
    mlflow.log_artifact("../models/light_gbm_model.joblib")

🏃 View run worried-shad-33 at: http://127.0.0.1:5000/#/experiments/1/runs/fbc46349c78840a2aad66068210d3bdf
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


#### Verify MLFlow Runs

In [62]:
import joblib
from pathlib import Path

model_dir = Path("../models")

rf_model = joblib.load(model_dir/"random_forest_model.joblib")
lgbm_model = joblib.load(model_dir/"light_gbm_model.joblib")
scaler = joblib.load(model_dir/"scaler.joblib")

print("Random Forest:", type(rf_model))
print("LightGBM:", type(lgbm_model))
print("Scaler:", type(scaler))
print("All three artifacts loaded successfully!")

Random Forest: <class 'sklearn.ensemble._forest.RandomForestRegressor'>
LightGBM: <class 'lightgbm.sklearn.LGBMRegressor'>
Scaler: <class 'sklearn.preprocessing._data.StandardScaler'>
All three artifacts loaded successfully!


#### Reproduce validation metrics

In [63]:
rf_y_v_pred = rf_loaded_model.predict(X_valid_scaled)
lgbm_y_v_pred = light_loaded_model.predict(X_valid_scaled)

In [64]:
rf_v_mae = mean_absolute_error(y_valid, rf_y_v_pred)
rf_v_rmse = np.sqrt(mean_squared_error(y_valid, rf_y_v_pred))
rf_v_r2 = r2_score(y_valid, rf_y_v_pred)

lgbm_v_mae = mean_absolute_error(y_valid, lgbm_y_v_pred)
lgbm_v_rmse = np.sqrt(mean_squared_error(y_valid, lgbm_y_v_pred))
lgbm_v_r2 = r2_score(y_valid, lgbm_y_v_pred)

In [65]:
print(rf_v_mae, rf_v_rmse, rf_v_r2)
print(lgbm_v_mae, lgbm_v_rmse, lgbm_r2)

18461.228505186267 56299.38222617333 0.8108962016009157
18642.077605294024 58044.31128233463 0.813841110369254


In [66]:
import mlflow
from mlflow.tracking import MlflowClient

print("Tracking URI:", mlflow.get_tracking_uri())

client = MlflowClient()

run_ids = {
    "Random Forest": "a3919db875794e66a769c0fadf4cb57e",
    "LightGBM": "fbc46349c78840a2aad66068210d3bdf"
}

for model_name, run_id in run_ids.items():
    run = client.get_run(run_id)

    print(f"\n--- {model_name} ---")
    print("Run ID:", run.info.run_id)
    print("Experiment ID:", run.info.experiment_id)
    print("Artifact URI:", run.info.artifact_uri)

    artifacts = client.list_artifacts(run_id)

    print("Artifacts:")
    for artifact in artifacts:
        print(" -", artifact.path)

Tracking URI: http://127.0.0.1:5000/

--- Random Forest ---
Run ID: a3919db875794e66a769c0fadf4cb57e
Experiment ID: 1
Artifact URI: mlflow-artifacts:/1/a3919db875794e66a769c0fadf4cb57e/artifacts
Artifacts:
 - random_forest_model.joblib
 - scaler.joblib

--- LightGBM ---
Run ID: fbc46349c78840a2aad66068210d3bdf
Experiment ID: 1
Artifact URI: mlflow-artifacts:/1/fbc46349c78840a2aad66068210d3bdf/artifacts
Artifacts:
 - light_gbm_model.joblib
 - scaler.joblib
